# resenh quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SharadhNaidu/resen/blob/main/notebooks/quickstart.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/SharadhNaidu/resen/blob/main/notebooks/quickstart.ipynb)

**resenh** brightens dark photos without blowing out lamps, amplifying noise or inventing colours. It combines a classical Retinex pipeline with a fine-tuned Retinexformer, and a quality controller fuses the two.

A GPU is optional: everything runs on CPU, just more slowly. On Kaggle, turn on *Internet* in the notebook settings so the model weights (6 MB) can download.

In [ ]:
%pip install -q "resenh[dl]"

In [ ]:
import resenh

resenh.info();  # versions, device (cuda / cpu) and cached weights

## 1. Enhance a photo

`enhance` accepts a file path, a URL, raw bytes, a PIL image or a NumPy array. Put the result on the last line of a cell to see before and after.

In [ ]:
url = "https://raw.githubusercontent.com/SharadhNaidu/resen/main/app/examples/lime_1.jpg"
res = resenh.enhance(url)
res

In [ ]:
print(res.summary())   # what the analyzer measured, the parameters it chose, how the controller fused

## 2. Look inside

Every stage is kept: illumination map, reflectance, gain, denoising, CLAHE, the classical and network outputs, and the final fusion.

In [ ]:
res.show(stages=True)

In [ ]:
res.decision   # quality-controller weights, guard scores (clipping, colour shift, edges, noise) and notes

## 3. Compare the three methods

In [ ]:
from resenh.viz import side_by_side

x = resenh.load_image(url)
outs = [(x, "input")] + [(resenh.enhance(x, mode=m).image, m) for m in ("classical", "dl", "hybrid")]
side_by_side(outs, height=300)

## 4. Your own photos

Upload a photo: in **Colab**, use the cell below; in **Kaggle**, add a dataset and point `path` at it, e.g. `/kaggle/input/<dataset>/photo.jpg`.

In [ ]:
try:
    from google.colab import files          # Colab: pick a file from your computer
    path = next(iter(files.upload()))
except ImportError:
    path = url                                # elsewhere: replace with your own path
res = resenh.enhance(path)
res.save("enhanced.png")
res

## 5. A whole folder, and the command line

In [ ]:
import os, urllib.request
os.makedirs("dark", exist_ok=True)
for name in ["dicm_3", "vv_5", "mef_2"]:
    urllib.request.urlretrieve(f"https://raw.githubusercontent.com/SharadhNaidu/resen/main/app/examples/{name}.jpg", f"dark/{name}.jpg")

written = resenh.enhance_folder("dark", "bright")   # same options as enhance(): mode=, device=, ...
written

In [ ]:
!resenh enhance dark -o bright_cli --mode hybrid

## Options at a glance

| argument | values | meaning |
|---|---|---|
| `mode` | `"auto"` (default), `"hybrid"`, `"dl"`, `"classical"` | `classical` needs no PyTorch |
| `preset` | `"robust"` (default), `"benchmark"` | robust adds a noise-gated pre-denoise for noisy phone photos |
| `device` | `"auto"`, `"cuda"`, `"cpu"`, `"mps"` | |
| `max_side` | `1280` | the network runs at most at this size; detail is carried back to full resolution |
| `tta` | `False` | ×8 self-ensemble: slightly better, 8× slower |

Code, results and documentation: https://github.com/SharadhNaidu/resen · Try it in the browser: https://resenh.sharadh.blog